# 09 — The island formula and the Page curve

**Spacetime Lab — Phase 9 concept + implementation notebook**

**This is the culmination.** Eight phases of preparatory work have brought us here:

1. **Phase 1**: Schwarzschild metric + horizon thermodynamics → the *Bekenstein-Hawking entropy* $S = A/(4 G_N)$
2. **Phase 2**: Penrose diagrams + causal structure
3. **Phase 3**: Kerr + symplectic geodesic integrator + Carter's constant
4. **Phase 4**: Horizon finders + the *Bardeen photon shadow* (the EHT geometry)
5. **Phase 5**: Quasinormal modes + ringdown → LIGO-style waveforms
6. **Phase 6**: Quantum information primitives → density matrices, von Neumann entropy, Schmidt decomposition
7. **Phase 7**: AdS/CFT foundations + *the simplest Ryu-Takayanagi check*, bit-exact
8. **Phase 8**: BTZ + Strominger 1998 + the *holographic phase transition*, all bit-exact
9. **Phase 9 (this notebook)**: The *island formula* and the *Page curve*

Phase 9 is the modern resolution of the **Hawking information paradox** — work from 2019-2020 by Penington, and independently by Almheiri, Engelhardt, Marolf and Maxfield, that finally explained how a black hole can emit Hawking radiation **unitarily**. The headline result is a curve: **the Page curve**. We will plot it, verify it bit-exactly, and ship the v1.0 release.

**What you will learn**

1. The Hawking information paradox in one paragraph
2. Don Page's 1993 prediction for the unitary curve
3. The island formula as a $\min$ over candidate quantum extremal surfaces
4. The Hartman-Maldacena 2013 calculation in eternal BTZ as the simplest closed-form example
5. **The Page curve, plotted** — rises linearly until the Page time, then saturates at $2 S_{BH}$
6. Why this is mathematically the same `min(...)` phase transition as the two-interval RT in Phase 8

**References**

- Hawking, *Particle creation by black holes*, Comm. Math. Phys. **43** 199 (1975)
- Page, *Information in black hole radiation*, Phys. Rev. Lett. **71** 3743 (1993), [arXiv:hep-th/9306083](https://arxiv.org/abs/hep-th/9306083)
- Hartman & Maldacena, [arXiv:1303.1080](https://arxiv.org/abs/1303.1080)
- Penington, [arXiv:1905.08255](https://arxiv.org/abs/1905.08255)
- Almheiri, Engelhardt, Marolf, Maxfield, [arXiv:1905.08762](https://arxiv.org/abs/1905.08762)
- Almheiri, Hartman, Maldacena, Shaghoulian, Tajdini, *The entropy of Hawking radiation*, Rev. Mod. Phys. **93** 035002 (2021), [arXiv:2006.06872](https://arxiv.org/abs/2006.06872) — the canonical review

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.metrics import BTZ
from spacetime_lab.holography import (
    hartman_maldacena_entropy,
    hartman_maldacena_growth_rate,
    island_saddle_entropy,
    page_curve,
    page_time,
    verify_page_curve_unitarity,
    brown_henneaux_central_charge,
)

plt.rcParams['figure.figsize'] = (8, 5)

## 1. The Hawking information paradox (1976)

In 1974-1976 Stephen Hawking calculated the spectrum of radiation emitted by a black hole formed from gravitational collapse. The radiation is **purely thermal**: a Planck distribution at temperature $T_H = \kappa/(2\pi)$. For Schwarzschild this is $T_H = 1/(8\pi M)$ — a number we computed all the way back in Phase 1.

Three properties of this thermal radiation:

1. **Pure thermal = maximum entropy at zero correlation.** Each emitted quantum is independent.
2. **The radiation entropy grows monotonically** as the BH evaporates: $S_\text{rad}(t)$ goes up forever, until the BH disappears.
3. **The BH entropy $S_{BH} = A/(4 G_N)$ shrinks** as the BH loses mass.

**Here is the paradox.** Quantum mechanics says: the *total* state of (BH + radiation) starts pure (the collapsing star), and unitary evolution preserves purity. The total entropy must remain *zero forever*, so the radiation entropy must always satisfy $S_\text{rad}(t) \le S_{BH}(t)$ — you can't have more entanglement than the BH still has degrees of freedom to be entangled with.

But Hawking's monotonically rising $S_\text{rad}$ exceeds the shrinking $S_{BH}$ at the **Page time** $t_P$. After this, the inequality is violated. Either Hawking's calculation is wrong, or quantum mechanics fails. **For 40+ years no one knew which.**

## 2. Page's 1993 prediction

Don Page derived the unitary answer purely from a typical-state argument, without knowing the microscopic mechanism:

- **Early times** ($S_\text{rad} < S_{BH}$): the radiation is the smaller subsystem, $S_\text{rad}(t)$ rises linearly with the number of emitted quanta.
- **Late times** ($S_\text{rad} > S_{BH}$): the BH is the smaller subsystem, $S_\text{rad}(t) = S_{BH}(t)$, *decreasing* as the BH shrinks.
- **Page time** $t_P$: the moment when the two cross. After this, the radiation entropy *starts to decrease*.
- **Final state**: $S_\text{rad} = 0$ when the BH has fully evaporated. All information is in the radiation correlations.

**The shape**: rises, peaks at $t_P$, falls back to zero. This is the **Page curve**. It is the unitary prediction.

## 3. The island formula (Penington 2019, AEMM 2019)

The breakthrough: **the radiation entropy is not just the von Neumann entropy of the bulk semi-classical state.** It is a `min` over candidate **quantum extremal surfaces** (QES):

$$S_\text{rad} = \min_X \, \text{ext}_X\!\left[\frac{\text{Area}(\partial X)}{4 G_N} + S_\text{semi-classical}(\text{Rad} \cup X)\right].$$

The candidate $X$ is an **island** — a region of the bulk inside the BH whose interior is added to the radiation when computing the entropy. There are two natural saddles:

- **Trivial saddle** ($X = \emptyset$): no island, just Hawking's calculation. Gives the rising thermal entropy.
- **Island saddle** ($X = $ region of the interior near the horizon): contributes an area term $\sim 2 S_{BH}$, plus a *small* semi-classical entropy because the island contains the partner modes that purify the Hawking radiation.

**The min picks the smaller one**: trivial early, island late. The crossover is the Page time. **This reproduces Page's curve from a purely gravitational calculation.**

Mathematically this is the **same `min(...)` structure as the two-interval RT phase transition we computed in Phase 8** — that's why Phase 8 was the right warm-up. The cross-ratio that controlled the two-interval transition becomes *time* in the island formula.

## 4. The Hartman-Maldacena 2013 setup

The cleanest closed-form example is the **eternal BTZ thermofield double**: an eternal AdS$_3$ black hole with two asymptotic boundaries (the "left" and "right" boundaries) connected through the bulk by an Einstein-Rosen bridge.

In this setup the two saddles have closed-form expressions:

- **Trivial / connected (HM) saddle**: a bulk geodesic that goes through the *growing* eternal-BH wormhole. As time advances the wormhole gets longer, so the geodesic length grows linearly. The corresponding entanglement entropy is

$$S_\text{HM}(t) = \frac{c}{3} \log\!\left[\frac{\beta}{\pi\epsilon}\cosh\!\left(\frac{2\pi t}{\beta}\right)\right].$$

At late times $\cosh(2\pi t/\beta) \sim \tfrac{1}{2}e^{2\pi t/\beta}$ and the entropy grows linearly with rate $\dot S_\text{HM} = 2\pi c/(3\beta)$.

- **Disconnected / island saddle**: two separate geodesics, one on each side of the eternal BH, both anchored to the bifurcation surface (the horizon). Each contributes $S_{BH}$ and the total is the constant $2 S_{BH} = \pi r_+/G_N$ — *independent of time*.

The Page curve is the minimum:

$$S_\text{Page}(t) = \min\!\left(S_\text{HM}(t),\ 2 S_{BH}\right).$$

For the eternal BH (which doesn't actually evaporate), the curve **rises linearly until $t_P$ and then saturates at $2 S_{BH}$** — it doesn't come back down. The qualitative *resolution of the paradox* is present: there exists a non-trivial saddle that prevents the entropy from growing forever.

## 5. Setting up an eternal BTZ black hole

In [ ]:
# Pick a moderate BTZ black hole
rp = 1.0       # horizon radius
L = 1.0        # AdS radius
G_N = 1.0      # Newton's constant
epsilon = 0.05  # UV cutoff (generous so we are in the standard regime)

bh = BTZ(horizon_radius=rp, ads_radius=L)
beta = bh.thermal_beta()
c = brown_henneaux_central_charge(radius=L, G_N=G_N)

S_BH = bh.bekenstein_hawking_entropy(G_N=G_N)
two_S_BH = island_saddle_entropy(rp, G_N=G_N)
rate = hartman_maldacena_growth_rate(c, beta)

print(f'BTZ black hole: r_+ = {rp}, L = {L}, G_N = {G_N}')
print(f'  Hawking temperature beta = {beta:.6f}')
print(f'  Brown-Henneaux central charge c = {c:.6f}')
print(f'  Bekenstein-Hawking entropy S_BH = {S_BH:.6f}')
print(f'  Island saddle 2 S_BH = {two_S_BH:.6f}')
print(f'  HM late-time growth rate dS/dt = 2 pi c / (3 beta) = {rate:.6f}')
print(f'  UV cutoff epsilon = {epsilon}')

## 6. The Page time

Numerically root-find the equation $S_\text{HM}(t_P) = 2 S_{BH}$ and compare against the late-time linear approximation $t_P \approx 3 \beta S_{BH}/(\pi c)$. The exact root differs from the approximation by log corrections.

In [ ]:
t_P = page_time(rp, L, epsilon, G_N=G_N)
t_P_approx = (3.0 * beta * S_BH) / (math.pi * c)

print(f'Page time (numerical):       t_P       = {t_P:.6f}')
print(f'Page time (late-T approx):   t_P_approx = {t_P_approx:.6f}')
print(f'Difference (log corrections): {abs(t_P - t_P_approx):.4f}')

# Verify continuity: at the Page time, the two saddles match exactly
S_hm_at_page = hartman_maldacena_entropy(t_P, c, beta, epsilon)
print(f'\nContinuity at the Page time:')
print(f'  S_HM(t_P) = {S_hm_at_page:.10f}')
print(f'  2 S_BH    = {two_S_BH:.10f}')
print(f'  residual: {abs(S_hm_at_page - two_S_BH):.2e}')

## 7. The Page curve — the headline plot of the entire project

The bulk RT prescription says: at each time $t$, take the minimum of $S_\text{HM}(t)$ and $2 S_{BH}$. Below the Page time the trivial saddle wins (rising linearly); above the Page time the island saddle wins (constant).

**This is the resolution of the Hawking information paradox.** The non-trivial saddle is what saves unitarity.

In [ ]:
times = np.linspace(0, 3 * t_P, 600)
S_hm_values = np.array([
    hartman_maldacena_entropy(t, c, beta, epsilon) for t in times
])
S_island_values = np.full_like(times, two_S_BH)
S_page_values = np.minimum(S_hm_values, S_island_values)

fig, ax = plt.subplots(figsize=(10, 6))

# Hawking saddle (the rising curve, the wrong answer)
ax.plot(times, S_hm_values, color='#888888', lw=1.5, ls='--',
        label=r'Hawking / trivial saddle $S_\mathrm{HM}(t)$')

# Island saddle (the constant)
ax.plot(times, S_island_values, color='#1a9a4a', lw=1.5, ls='--',
        label=r'island saddle $2 S_{BH}$ (constant)')

# Page curve (the min — the right answer)
ax.plot(times, S_page_values, color='#0050a0', lw=3.0,
        label=r'**Page curve** $\min(S_\mathrm{HM},\, 2 S_{BH})$')

# Mark the Page time
ax.axvline(t_P, color='#a40000', lw=1.5, ls=':',
           label=f'Page time $t_P \\approx {t_P:.3f}$')
ax.axhline(two_S_BH, color='#1a9a4a', lw=0.8, alpha=0.4)

ax.set_xlabel(r'boundary time $t$ (geometric units)')
ax.set_ylabel(r'radiation entropy $S_\mathrm{rad}(t)$ (nats)')
ax.set_title('The Page curve from the island formula\n'
             '(eternal BTZ, Hartman-Maldacena 2013)')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 3 * t_P)
plt.tight_layout()
plt.show()

**Read the plot like this:**

- The **dashed grey curve** is what Hawking would have computed: the trivial saddle, rising linearly forever. This is the calculation that *appears* to violate unitarity.
- The **dashed green line** is the island saddle: a constant at $2 S_{BH}$. For the eternal BH it doesn't fall, because the BH doesn't actually shrink. For an evaporating BH (which we don't model here) this line would also fall, giving the falling tail of Page's curve.
- The **solid blue curve** is the Page curve: the `min` of the two. **This is the right answer.** It is the prediction of the island formula, derived from a *purely gravitational* calculation. It saturates at $2 S_{BH}$ instead of growing forever, resolving the paradox.
- The **red dotted vertical line** marks the Page time $t_P$, where the two saddles cross. *The transition is sharp.* At $t_P$ the bulk RT prescription switches from picking the trivial saddle to picking the island saddle.

**This is the simplest non-trivial demonstration of the island formula in eternal AdS.** It captures the qualitative resolution: a non-trivial saddle exists, it eventually beats the Hawking saddle, and the entropy stops growing. The full evaporating-BH version (Penington 2019, AEMM 2019) gives a curve that actually goes back down to zero — but it requires more machinery (coupling the BH to a non-gravitational reservoir) and the qualitative point is the same.

## 8. Why this is the same `min(...)` as the Phase 8 two-interval phase transition

Compare to the Phase 8 two-interval mutual information plot:

- **Phase 8**: as the gap between two boundary intervals grows, the bulk RT prescription switches from the *connected* configuration (one geodesic linking the four endpoints) to the *disconnected* configuration (two separate geodesics). The mutual information has a sharp non-analytic kink at the cross-ratio $x = 1/2$.
- **Phase 9 (here)**: as time advances in the eternal BH, the bulk RT prescription switches from the *trivial* (HM) saddle to the *island* (disconnected) saddle. The radiation entropy has a sharp non-analytic kink at the Page time.

**Same mathematical structure, completely different physical context.** The cross-ratio in Phase 8 plays the role of time in Phase 9. The connected saddle in Phase 8 corresponds to the HM (wormhole-traversing) saddle in Phase 9. The disconnected saddle in Phase 8 corresponds to the island saddle in Phase 9.

This is what made Phase 8 the right preparation for Phase 9: the same $\min(\mathcal{L}^A, \mathcal{L}^B)/(4 G_N)$ machinery, applied to a different physical question, gives a different result that *also* resolves a longstanding puzzle. **The holographic dictionary is unreasonably effective.**

## 9. The Page curve at multiple BH sizes — an instructive comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for rp_val, color in [(0.5, '#0050a0'), (1.0, '#1a9a4a'), (2.0, '#c64a0b'), (4.0, '#7030a0')]:
    bh_local = BTZ(horizon_radius=rp_val, ads_radius=L)
    beta_local = bh_local.thermal_beta()
    two_S_BH_local = island_saddle_entropy(rp_val, G_N=G_N)
    t_P_local = page_time(rp_val, L, epsilon, G_N=G_N)
    
    times_local = np.linspace(0, 1.2 * t_P_local, 400)
    S_hm_local = np.array([
        hartman_maldacena_entropy(t, c, beta_local, epsilon) for t in times_local
    ])
    S_page_local = np.minimum(S_hm_local, two_S_BH_local)
    
    ax.plot(times_local, S_page_local, color=color, lw=2.5,
            label=f'$r_+ = {rp_val}$, $t_P = {t_P_local:.3f}$, $2 S_{{BH}} = {two_S_BH_local:.3f}$')
    ax.axvline(t_P_local, color=color, lw=0.8, ls=':', alpha=0.5)

ax.set_xlabel(r'boundary time $t$')
ax.set_ylabel(r'radiation entropy $S_\mathrm{rad}(t)$ (nats)')
ax.set_title('Page curves for several BTZ black hole sizes\n'
             '(larger BH → larger saturation entropy → later Page time)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Notice three features:

1. **Larger BHs have larger saturation entropies** (the $2 S_{BH}$ plateau is higher), because they have more degrees of freedom to hide information in.
2. **Larger BHs have later Page times**, because their plateau is higher and the linearly-growing HM curve takes longer to reach it.
3. **All four curves have the same morphology**: linear rise, sharp kink, flat saturation. The shape is *universal* — only the scale varies. This is exactly what one expects from a holographic prescription that depends only on the BH parameters via the central charge and the Bekenstein-Hawking entropy.

## 10. Phase 9 gate checks

In [ ]:
# (1) HM saddle at t = 0 has the canonical closed form
S_hm_0 = hartman_maldacena_entropy(0.0, c, beta, epsilon)
expected_t0 = (c / 3.0) * math.log(beta / (math.pi * epsilon))
assert math.isclose(S_hm_0, expected_t0, abs_tol=1e-12)

# (2) HM late-time growth rate matches dS/dt = 2 pi c / (3 beta)
rate_predicted = hartman_maldacena_growth_rate(c, beta)
rate_numerical = (
    hartman_maldacena_entropy(1001, c, beta, epsilon)
    - hartman_maldacena_entropy(1000, c, beta, epsilon)
)
assert math.isclose(rate_numerical, rate_predicted, abs_tol=1e-9)

# (3) Island saddle equals 2 S_BH = pi r_+ / G_N
assert math.isclose(island_saddle_entropy(rp, G_N=G_N), math.pi * rp / G_N)

# (4) Continuity at the Page time: S_HM(t_P) = 2 S_BH
S_hm_at_P = hartman_maldacena_entropy(t_P, c, beta, epsilon)
assert math.isclose(S_hm_at_P, two_S_BH, abs_tol=1e-9)

# (5) Page curve is in the trivial phase before t_P and island after
_, phase_before = page_curve(0.5 * t_P, rp, L, epsilon, G_N=G_N)
_, phase_after = page_curve(2.0 * t_P, rp, L, epsilon, G_N=G_N)
assert phase_before == 'trivial'
assert phase_after == 'island'

# (6) After the Page time the curve is exactly constant at 2 S_BH
for t in [t_P * 1.5, t_P * 2.0, t_P * 5.0, t_P * 100.0]:
    S_late, _ = page_curve(t, rp, L, epsilon, G_N=G_N)
    assert math.isclose(S_late, two_S_BH, abs_tol=1e-12)

# (7) Page curve is monotonically non-decreasing
diffs = np.diff(S_page_values)
assert np.all(diffs >= -1e-10)

# (8) The full verification gate function is happy
diag = verify_page_curve_unitarity(rp, L, epsilon, G_N=G_N)
assert diag['trivial_at_early']
assert diag['island_at_late']
assert diag['monotonic']
assert diag['continuity_residual'] < 1e-9

# (9) Page time is independent of G_N (a non-obvious feature of the
#     holographic setup: c and S_BH both scale as 1/G_N, so G_N cancels
#     in the equation S_HM(t_P) = 2 S_BH)
t_P_GN1 = page_time(rp, L, epsilon, G_N=1.0)
t_P_GN2 = page_time(rp, L, epsilon, G_N=2.0)
assert math.isclose(t_P_GN1, t_P_GN2, abs_tol=1e-9)

print('ALL PHASE 9 GATE CHECKS PASSED')

---

## The end of the roadmap — Spacetime Lab v1.0

**This is the v1.0 milestone.** Eight phases of preparatory work, plus this one, constitute the complete arc of the project:

| Phase | Concept | Headline result |
|---|---|---|
| 1 | Schwarzschild metric | Bekenstein-Hawking $S = A/(4 G_N)$ |
| 2 | Penrose diagrams | Causal structure of black holes |
| 3 | Kerr + symplectic integrator | Carter's irreducible Killing tensor |
| 4 | Horizon finders + photon shadow | Bardeen 1973 EHT geometry |
| 5 | Quasinormal modes | Schwarzschild QNM verified vs Berti et al |
| 6 | Quantum information | von Neumann entropy + Schmidt decomposition |
| 7 | AdS/CFT foundations | Ryu-Takayanagi bit-exact vs Calabrese-Cardy |
| 8 | BTZ + holographic phase transition | Strominger 1998 BTZ-Cardy match, Headrick mutual info |
| **9** | **Island formula** | **Page curve from the trivial-vs-island saddle transition** |

And the punchline: **the factor of $1/(4 G_N)$ appears in every single phase**. It is the Bekenstein-Hawking factor in Phases 1, 3, 4. It is the Brown-Henneaux relation $c = 3L/(2 G_N)$ in Phase 7. It is the Cardy-Strominger derivation in Phase 8. It is the area term in the island formula in Phase 9. **The same number ties classical horizon area to quantum entanglement entropy across every level of the holographic dictionary.**

This is what holography looks like when you can run it.

## Honest scope notes for Phase 9

We deliberately implemented the **simplest** version of the island formula:

- **Eternal BTZ**, not an evaporating BH. The eternal-BH Page curve **rises and saturates** but doesn't fall, because the BH doesn't actually shrink. The full evaporating Page curve (Penington 2019, AEMM 2019) requires coupling to a non-gravitational reservoir and would be a v1.1 patch.
- **Hartman-Maldacena 2013 closed form**, not a numerical extremal-surface finder. The closed form makes the verification bit-exact.
- **No quantum extremal surface formalism**. The island here is identified by hand (it lives at the bifurcation surface). The QES formalism that derives the island location from a variational principle is an abstraction we don't need for this simplest example.
- **No replica wormhole derivation**. The 2019-2020 papers derive the island formula from a Euclidean gravitational path integral via the replica trick on multiple disconnected gravitational saddles. That is *the proof*, but it is an analytical argument that doesn't translate naturally into runnable Python.

These are honest scope decisions. The qualitative resolution of the paradox — *there is a non-trivial saddle that prevents the radiation entropy from growing forever* — is fully present in this notebook. That is the v1.0 milestone.